# Gemma-4-e2b: Minimal PA-CCS Pipeline

This notebook runs the same PA-CCS workflow used in the project codebase:

1. Load a dataset (`polarity_raw`, `single`, or `paired`)
2. Extract hidden states for `Yes.` / `No.` suffix prompts
3. Train layer-wise PA-CCS probes
4. Save + inspect `ccs_summary.csv`
5. Visualize layer-wise accuracy and best-layer comparison

Datasets covered here:
- Mixed polarity dataset (`polarity_raw`)
- ToxiGen single dataset (`single`)
- ToxiGen paired dataset (`paired`, if present)

The notebook is intentionally compact and reuses cached embeddings to avoid repeated extraction.

In [1]:
from pathlib import Path
import json
import sys

import numpy as np
import pandas as pd
import torch
import matplotlib.pyplot as plt
from IPython.display import display

# Make local package importable without reinstall on every notebook restart.
PROJECT_ROOT = (Path.cwd().resolve().parent if Path.cwd().name == "notebooks" else Path.cwd().resolve())
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from latent_alignment.data import load_dataset
from latent_alignment.extract import load_embeddings, load_hf_model, extract_texts, save_embeddings
from latent_alignment.ccs import ProbeConfig, train_ccs_layers, summarize_results

print(f"Project root: {PROJECT_ROOT}")

ModuleNotFoundError: No module named 'latent_alignment'

In [ ]:
# ---- Experiment config ----
MODEL_NAME = "google/gemma-4-e2b"  # change if your checkpoint id/path differs
MODEL_KIND = "decoder"
STRATEGY = "last-token"
DTYPE = "bfloat16"                  # "auto", "float16", "bfloat16", "float32"
MAX_LENGTH = 512
DEVICE = None
TRUST_REMOTE_CODE = False

NORMALIZING = "l2,median"

DATA_ROOT = PROJECT_ROOT / "data" / "polarity_probing" / "raw"
MIXED_DATASET = DATA_ROOT / "mixed_dataset.csv"
TOXIGEN_SINGLE_DATASET = DATA_ROOT / "toxigen_annotated_test.csv"
TOXIGEN_PAIRED_DATASET = DATA_ROOT / "toxigen_annotated_test_paired.csv"

RUNS_ROOT = PROJECT_ROOT / "runs"
RUN_MIXED = RUNS_ROOT / "gemma4_e2b_mixed"
RUN_TOXIGEN_SINGLE = RUNS_ROOT / "gemma4_e2b_toxigen_single"
RUN_TOXIGEN_PAIRED = RUNS_ROOT / "gemma4_e2b_toxigen_paired"

# Probe defaults (same as CLI defaults)
PROBE_CONFIG = ProbeConfig(
    nepochs=1500,
    ntries=10,
    lr=0.015,
    batch_size=-1,
    weight_decay=0.01,
    lambda_classification=0.0,
    normalizing=NORMALIZING,
    seed=0,
)

print(f"Runs root: {RUNS_ROOT}")
print(f"Mixed exists: {MIXED_DATASET.exists()} | {MIXED_DATASET}")
print(f"ToxiGen single exists: {TOXIGEN_SINGLE_DATASET.exists()} | {TOXIGEN_SINGLE_DATASET}")
print(f"ToxiGen paired exists: {TOXIGEN_PAIRED_DATASET.exists()} | {TOXIGEN_PAIRED_DATASET}")

In [2]:
def ensure_embeddings(
    *,
    dataset_path: Path,
    dataset_format: str,
    embeddings_path: Path,
    model,
    tokenizer,
    device,
):
    """Load cached embeddings or extract + cache them."""
    dataset = load_dataset(dataset_path, dataset_format=dataset_format)

    if embeddings_path.exists():
        positive, negative = load_embeddings(embeddings_path)
        return dataset, positive, negative

    positive = extract_texts(
        dataset.positive_texts,
        model,
        tokenizer,
        get_all_layers=True,
        strategy=STRATEGY,
        model_kind=MODEL_KIND,
        device=device,
        max_length=MAX_LENGTH,
    )
    negative = extract_texts(
        dataset.negative_texts,
        model,
        tokenizer,
        get_all_layers=True,
        strategy=STRATEGY,
        model_kind=MODEL_KIND,
        device=device,
        max_length=MAX_LENGTH,
    )

    if positive.ndim == 2:
        positive = positive[:, None, :]
        negative = negative[:, None, :]

    embeddings_path.parent.mkdir(parents=True, exist_ok=True)
    save_embeddings(embeddings_path, positive, negative)
    return dataset, positive, negative


def run_pa_ccs(
    *,
    output_dir: Path,
    dataset_path: Path,
    dataset_format: str,
    model,
    tokenizer,
    device,
    probe_config: ProbeConfig = PROBE_CONFIG,
):
    """Run full PA-CCS on one dataset and save artifacts into output_dir."""
    output_dir.mkdir(parents=True, exist_ok=True)

    embeddings_path = output_dir / "embeddings.npz"
    dataset, positive, negative = ensure_embeddings(
        dataset_path=dataset_path,
        dataset_format=dataset_format,
        embeddings_path=embeddings_path,
        model=model,
        tokenizer=tokenizer,
        device=device,
    )

    train_idx, test_idx = dataset.train_test_indices(
        test_size=0.15,
        random_state=71,
        stratify=True,
    )

    results = train_ccs_layers(
        positive,
        negative,
        dataset.labels,
        train_idx,
        test_idx,
        config=probe_config,
        opposite_indices=dataset.opposite_indices,
        device=device,
    )

    summary_df = pd.DataFrame(summarize_results(results))
    summary_df.to_csv(output_dir / "ccs_summary.csv", index=False)

    full_results = {
        f"layer_{layer}_{key}": value
        for layer, row in results.items()
        for key, value in row.items()
    }
    np.savez_compressed(output_dir / "ccs_full_results.npz", **full_results)

    metadata = {
        "dataset_path": str(dataset_path),
        "dataset_format": dataset_format,
        "train_idx": train_idx.tolist(),
        "test_idx": test_idx.tolist(),
        "probe_config": probe_config.__dict__,
    }
    (output_dir / "metadata.json").write_text(json.dumps(metadata, indent=2), encoding="utf-8")

    return summary_df.sort_values("accuracy", ascending=False).reset_index(drop=True)

NameError: name 'PROBE_CONFIG' is not defined

In [3]:
model, tokenizer, device = load_hf_model(
    MODEL_NAME,
    model_kind=MODEL_KIND,
    device=DEVICE,
    dtype=DTYPE,
    trust_remote_code=TRUST_REMOTE_CODE,
)
print(f"Model loaded on: {device}")
print(f"CUDA available: {torch.cuda.is_available()}")

NameError: name 'load_hf_model' is not defined

## Mixed dataset (`polarity_raw`)

In [4]:
mixed_summary = run_pa_ccs(
    output_dir=RUN_MIXED,
    dataset_path=MIXED_DATASET,
    dataset_format="polarity_raw",
    model=model,
    tokenizer=tokenizer,
    device=device,
)

display(mixed_summary.head(10))
print(f"Saved mixed run: {RUN_MIXED}")

NameError: name 'run_pa_ccs' is not defined

## ToxiGen (`single` and optional `paired`)

You currently have two ToxiGen files:

- `toxigen_annotated_test.csv` -> **single** format (one statement per row)
  - gives `accuracy` + `silhouette`
  - `polar_consistency_mean` / `contradiction_index_mean` are `NaN`
- `toxigen_annotated_test_paired.csv` -> **paired** format (toxic statement + benign rewrite)
  - gives the full PA-CCS metric set, including polarity-aware metrics

In [5]:
PREPARE_TOXIGEN_IF_MISSING = False

if not TOXIGEN_SINGLE_DATASET.exists() and PREPARE_TOXIGEN_IF_MISSING:
    from latent_alignment.toxigen import build_toxigen_dataframe

    TOXIGEN_SINGLE_DATASET.parent.mkdir(parents=True, exist_ok=True)
    toxigen_df = build_toxigen_dataframe(
        config="annotated",
        split="test",
        toxic_threshold=3.0,
    )
    toxigen_df.to_csv(TOXIGEN_SINGLE_DATASET, index=False)
    print(f"Prepared ToxiGen single CSV at: {TOXIGEN_SINGLE_DATASET}")

NameError: name 'TOXIGEN_DATASET' is not defined

In [6]:
toxigen_single_summary = None
toxigen_paired_summary = None

if TOXIGEN_SINGLE_DATASET.exists():
    toxigen_single_summary = run_pa_ccs(
        output_dir=RUN_TOXIGEN_SINGLE,
        dataset_path=TOXIGEN_SINGLE_DATASET,
        dataset_format="single",
        model=model,
        tokenizer=tokenizer,
        device=device,
    )
    print(f"Saved toxigen single run: {RUN_TOXIGEN_SINGLE}")
    display(toxigen_single_summary.head(10))
else:
    print(f"ToxiGen single file not found: {TOXIGEN_SINGLE_DATASET}")

if TOXIGEN_PAIRED_DATASET.exists():
    toxigen_paired_summary = run_pa_ccs(
        output_dir=RUN_TOXIGEN_PAIRED,
        dataset_path=TOXIGEN_PAIRED_DATASET,
        dataset_format="paired",
        model=model,
        tokenizer=tokenizer,
        device=device,
    )
    print(f"Saved toxigen paired run: {RUN_TOXIGEN_PAIRED}")
    display(toxigen_paired_summary.head(10))
else:
    print(f"ToxiGen paired file not found: {TOXIGEN_PAIRED_DATASET}")

NameError: name 'TOXIGEN_DATASET' is not defined

In [ ]:
def best_layer_row(df: pd.DataFrame, name: str) -> dict:
    best = df.loc[df["accuracy"].idxmax()]
    return {
        "run": name,
        "best_layer": int(best["layer"]),
        "best_accuracy": float(best["accuracy"]),
        "best_silhouette": float(best["silhouette"]),
        "polar_consistency_mean": float(best["polar_consistency_mean"])
        if pd.notna(best["polar_consistency_mean"])
        else np.nan,
        "contradiction_index_mean": float(best["contradiction_index_mean"])
        if pd.notna(best["contradiction_index_mean"])
        else np.nan,
    }

summaries = {"mixed": mixed_summary}
if toxigen_single_summary is not None:
    summaries["toxigen_single"] = toxigen_single_summary
if toxigen_paired_summary is not None:
    summaries["toxigen_paired"] = toxigen_paired_summary

report = pd.DataFrame([best_layer_row(df, name) for name, df in summaries.items()])
display(report)

# --- minimal visualizations ---
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

for name, df in summaries.items():
    axes[0].plot(df["layer"], df["accuracy"], marker="o", linewidth=1.5, label=name)
axes[0].axhline(0.5, linestyle="--", color="gray", linewidth=1)
axes[0].set_title("PA-CCS accuracy by layer")
axes[0].set_xlabel("layer")
axes[0].set_ylabel("accuracy")
axes[0].legend()
axes[0].grid(alpha=0.3)

axes[1].bar(report["run"], report["best_accuracy"])
axes[1].set_title("Best accuracy per run")
axes[1].set_ylabel("accuracy")
axes[1].set_ylim(0, 1)
axes[1].grid(axis="y", alpha=0.3)

plt.tight_layout()
plt.show()

print(f"Saved run folders under: {RUNS_ROOT}")

NameError: name 'mixed_summary' is not defined